# 054 — The Keras Functional API

The Sequential API can only express a straight chain. Making layers **callable**
removes that limit: `Dense(64)(previous)` — the trailing call *is* the wiring.

Measured: age and gender from one face with a shared trunk costs
**24,152,384** parameters against **38,867,072** for two separate models.

| Part | What we check |
|---|---|
| A | where the straight chain breaks |
| B | the shared-trunk saving, computed |
| C | why `loss_weights` exists - a units problem, not a priority problem |
| D | a two-branch forward pass by hand |

## Part A — What Sequential cannot say

Neither of these is exotic. Both are ordinary product requirements.

In [ ]:
cases = [("two outputs from one input", "age and gender from one face"),
         ("two inputs to one output", "an image plus its tabular metadata"),
         ("a branch that rejoins", "two filter sizes concatenated"),
         ("a skip connection", "ResNet - lesson 062")]
for shape, example in cases:
    print(f"  {shape:<28} {example}")
print()
print("A Sequential model is a list of layers, and a list has no branches.")
print("The Functional API writes the same two layers as:")
print("    inputs = Input(shape=(10,));  x = Dense(64)(inputs)")
print("and the trailing parenthesis is the entire difference.")
assert len(cases) == 4

## Part B — What a shared trunk saves

In [ ]:
CONV_BASE = 14_714_688          # VGG16, include_top=False
INPUT = 200

size = INPUT
for _ in range(5):              # VGG16 halves the spatial size five times
    size //= 2
flat = size * size * 512
head = flat * 256 + 256

separate = 2 * CONV_BASE + 2 * head
shared = CONV_BASE + 2 * head
print(f"conv base output     {size}x{size}x512 = {flat:,}")
print(f"one head             {head:>12,}")
print(f"two separate models  {separate:>12,}")
print(f"one trunk, two heads {shared:>12,}   ({1 - shared / separate:.0%} fewer)")
assert separate == 38_867_072 and shared == 24_152_384
assert round((1 - shared / separate) * 100) == 38
print()
print("And the shared version convolves each image ONCE, not twice - the")
print("saving at prediction time is larger than the parameter saving.")

## Part C — Why `loss_weights` is not about priorities

Age is measured in years; gender loss in nats. Added together unweighted, the
gender term is a few percent of the total, so gradient descent barely trains
on it.

In [ ]:
GENDER_BCE = 0.35               # a typical converged binary cross-entropy
for age_mae in (4.0, 8.0, 15.0):
    total_loss = age_mae + GENDER_BCE
    print(f"age MAE {age_mae:>4.1f} yr + gender BCE {GENDER_BCE}  ->  total "
          f"{total_loss:>5.2f},  gender is {GENDER_BCE / total_loss:>5.1%}")
share = GENDER_BCE / (15.0 + GENDER_BCE)
assert share < 0.03
print()
print("Nothing here says age matters more. Years are simply bigger numbers")
print("than nats. loss_weights={'age': 0.1, 'gender': 1.0} puts the two on a")
print("comparable footing - it is a units problem, not a priority problem.")
weighted = 0.1 * 15.0 + 1.0 * GENDER_BCE
print(f"weighted: 0.1*15.0 + 1.0*0.35 = {weighted:.2f}, gender is "
      f"{GENDER_BCE / weighted:.0%} of it")
assert GENDER_BCE / weighted > 0.15

## Part D — Two branches and a merge, by hand

A tiny functional graph: two inputs of different widths, each through its own
dense layer, concatenated, then one output.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
image_features, meta_features, branch, out_units = 16, 4, 8, 1

W_img, b_img = rng.normal(0, 0.3, (image_features, branch)), np.zeros(branch)
W_meta, b_meta = rng.normal(0, 0.3, (meta_features, branch)), np.zeros(branch)
W_out, b_out = rng.normal(0, 0.3, (2 * branch, out_units)), np.zeros(out_units)

x_img = rng.normal(0, 1, image_features)
x_meta = rng.normal(0, 1, meta_features)

a = np.maximum(0, x_img @ W_img + b_img)        # branch 1
b = np.maximum(0, x_meta @ W_meta + b_meta)     # branch 2
merged = np.concatenate([a, b])                 # Concatenate()([a, b])
pred = 1 / (1 + np.exp(-(merged @ W_out + b_out)))

print(f"branch 1 output {a.shape}, branch 2 output {b.shape}")
print(f"concatenated    {merged.shape}   <- (8,) + (8,) = (16,)")
print(f"prediction      {pred[0]:.4f}")
assert merged.shape == (16,)

params = W_img.size + b_img.size + W_meta.size + b_meta.size + W_out.size + b_out.size
print(f"\nparameters: {params}")
assert params == 16 * 8 + 8 + 4 * 8 + 8 + 16 * 1 + 1
print("Two inputs of different widths met in one model. Sequential cannot")
print("express this at all.")

## What this notebook established

- The Functional API's whole mechanism is that a layer is callable, which is
  what makes branches and merges expressible.
- Age and gender from one face: **38,867,072** parameters as two models against
  **24,152,384** with a shared trunk — **38% fewer**, and one convolution pass
  instead of two.
- Unweighted, a gender cross-entropy of 0.35 is under **3%** of a loss whose
  other term is an age MAE in years. `loss_weights` fixes the units.
- Concatenating `(8,)` and `(8,)` gives `(16,)`; two inputs of different widths
  can meet in one model.

## Exercises

1. Add a third head (ethnicity, 5 classes) to the shared trunk. What does it
   cost, and why so little?
2. At what input size does the shared-trunk saving fall below 30%?
3. Write the Part D graph as two separate models and count the parameters.
   Where exactly does the duplication go?